# Predictive Price Modeling
This notebook trains a baseline `RandomForestRegressor` to predict hourly `NO2` electricity prices (`nok_per_kwh`) from weather and calendar features, tracks parameters and evaluation metrics in MLflow, and writes feature importances and model metrics to Gold tables for the dashboard.

### Key design choices in this notebook
* **Reading from joined Silver (`silver.weather_price_hourly`):** Uses the hourly joined dataset rather than the aggregated Gold bands so the model can learn from individual hourly observations. Rows with missing weather values are dropped explicitly via `dropna()` since Silver does not impute missing sensor data.
* **Weather + calendar features:** Combines physical weather drivers (`air_temperature_c`, `wind_speed_mps`) with time features (`hour` of day and `month` of year), because electricity prices depend heavily on both daily consumption peaks and seasonal heating demand.
* **MLflow experiment tracking:** Logs hyperparameters (`n_estimators=100`, `max_depth=10`), evaluation metrics (`mae`, `rmse`, `r2`), and the trained scikit-learn model artifact under the active user's workspace experiment path.
* **Saving outputs to Gold (`model_feature_importance` and `model_performance_metrics`):** Persists feature importances and test metrics into Unity Catalog Gold tables so the Lakeview dashboard can display model performance and feature drivers alongside the price charts.

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, r2_score, root_mean_squared_error)
from sklearn.model_selection import train_test_split

# Dynamically resolves both the active Unity Catalog (current_catalog()) and the active workspace user (current_user()) for the MLflow experiment path.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
username = spark.sql("SELECT current_user()").first()[0]

### Feature engineering, model training, and MLflow tracking
Loads `silver.weather_price_hourly`, drops hours with missing temperature or wind speed, extracts `hour` and `month` from `time_start_utc`, splits into 80% train / 20% test sets, trains a `RandomForestRegressor`, and logs parameters, metrics (`MAE`, `RMSE`, `R2`), and the model to MLflow.


In [0]:
# Load hourly joined Silver data into pandas and drop rows with missing weather values
df = spark.table(f"{CATALOG}.silver.weather_price_hourly").toPandas()

# Drop rows with missing values in key features
df = df.dropna(
    subset=["air_temperature_c", "wind_speed_mps", "nok_per_kwh"]
)

# Extract hour of day and month of year to capture daily and seasonal demand patterns
df["time_start_utc"] = pd.to_datetime(df["time_start_utc"])
df["hour"] = df["time_start_utc"].dt.hour
df["month"] = df["time_start_utc"].dt.month

# Define predictor features (X) and target variable (y)
features = ["air_temperature_c", "wind_speed_mps", "hour", "month"]
X = df[features]
y = df["nok_per_kwh"]

# Split into 80% training and 20% test data with a fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Set MLflow experiment path dynamically and log parameters, metrics, and model artifact
mlflow.set_experiment(f"/Users/{username}/weather_price_prediction")

with mlflow.start_run(run_name="random_forest_price_baseline"):
  # Train Random Forest Regressor
  n_estimators = 100
  max_depth = 10
  model = RandomForestRegressor(
      n_estimators=n_estimators, max_depth=max_depth, random_state=42
  )
  model.fit(X_train, y_train)

  # Predictions & Evaluation
  predictions = model.predict(X_test)
  mae = mean_absolute_error(y_test, predictions)
  rmse = root_mean_squared_error(y_test, predictions)
  r2 = r2_score(y_test, predictions)

  # Log parameters, metrics, and model to MLflow
  mlflow.log_param("n_estimators", n_estimators)
  mlflow.log_param("max_depth", max_depth)
  mlflow.log_metric("mae", mae)
  mlflow.log_metric("rmse", rmse)
  mlflow.log_metric("r2", r2)
  mlflow.sklearn.log_model(model, "model")

  print(f"Model trained successfully! MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

### Save feature importances and evaluation metrics to Gold
Writes the Random Forest feature importances (`gold.model_feature_importance`) and test set evaluation metrics (`gold.model_performance_metrics`) to Delta tables so they can be queried directly by the Lakeview dashboard.


In [0]:
# Extract feature importance scores from the trained Random Forest and overwrite Gold table
fi_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})
display(fi_df)

(
    spark.createDataFrame(fi_df)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.model_feature_importance")
)

In [0]:
# Save test set evaluation metrics (MAE, RMSE, R2) to Gold for dashboard reporting
metrics_df = pd.DataFrame([{
    "mae": float(mae),
    "rmse": float(rmse),
    "r2": float(r2)
}])
display(metrics_df)

(
    spark.createDataFrame(metrics_df)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.gold.model_performance_metrics")
)